# DS2002 · Data Cleaning (types, missing, duplicates)

**Lecture — 2026-09-21 · Fall 2026**  
**Class time:** 45 minutes

---

## Cleaning is not the part before the work

Every step so far assumed the data was already usable. It never is. Surveys of working data scientists put the share of time spent on cleaning somewhere north of half, and the reason is not that it is tedious — it is that **cleaning decisions change answers.**

We saw that in week one: normalizing one product name moved a poncho total by forty percent. Today we do the whole pipeline on eight deliberately awful rows, and at the end we measure how much the cleaning moved the number we care about.

Eight rows is on purpose. You can hold eight rows in your head, so you can tell whether each fix did what you meant. Then you apply the same sequence to three hundred thousand.

In [ ]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

### Take inventory before you touch anything

Three commands, every time, in this order. They tell you what kind of mess you have before you start guessing.

In [ ]:
print('shape:', df.shape)
print()
print(df.dtypes)
print()
print('nulls per column:')
print(df.isnull().sum())
print()
print('exact duplicate rows:', df.duplicated().sum())

Read what those are telling you. `qty` and `price` came in as `object`, which means **text** — so the file contains something non-numeric in both columns. There is one null in `item` and one in `ts`. And one row is an exact duplicate of another.

Before the fixes, write down the number you are trying to get right. Otherwise you have no way to tell whether cleaning helped.

In [ ]:
# The question: how much revenue did we take, and how many units moved?
# We cannot even compute it yet -- price is text. That is itself the first finding.
print(df['price'].tolist())
print(df['qty'].tolist())

### Fix 1 — duplicates, and which kind you mean

`drop_duplicates()` removes rows identical across **every** column. That is the safe version and it is what we want here: rows 0 and 1 are the same transaction exported twice.

In [ ]:
clean = df.drop_duplicates().copy()
print('rows:', len(df), '->', len(clean))

Be careful with the other kind. `drop_duplicates(subset=['order_id'])` keeps one row per order id — which is right if ids should be unique, and destroys your data if an order can legitimately have several line items. Look before you choose.

In [ ]:
# Here, order_id 1 appeared twice because of the double export -- already handled.
# But note what subset= would do if orders had multiple items each:
print('unique order_ids:', clean['order_id'].nunique(), 'of', len(clean), 'rows')

### Fix 2 — numbers that arrived as text

`price` has dollar signs. Strip them and convert. Real files also bring commas (`'1,240.00'`) and parentheses for negatives (`'(45.00)'`), so build the habit of stripping everything that is not part of a number.

In [ ]:
clean['price'] = (clean['price'].astype(str)
                  .str.replace('$', '', regex=False)
                  .str.replace(',', '', regex=False)
                  .str.strip()
                  .astype(float))
print(clean['price'].dtype)
clean[['order_id', 'item', 'price']]

### Fix 3 — coercion, and the decision it forces

`qty` contains the string `'NULL'`. `astype(int)` would just crash. `pd.to_numeric` with `errors='coerce'` turns anything unparseable into `NaN` — which does not solve the problem, it *surfaces* it. Now you have to decide what a missing quantity means.

In [ ]:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')
print(clean[['order_id', 'item', 'qty']])
print()
print('missing qty:', clean['qty'].isna().sum())
print('negative qty:', (clean['qty'] < 0).sum())

Two separate problems, two separate decisions:

**Missing quantity.** Dropping the row loses a real sale. Filling with 1 invents data. Filling with the median invents data and hides it. Here, dropping is defensible because one row of eight is a small, reportable loss — but you report it.

**Negative quantity.** A `-3` is almost certainly a refund. Whether it belongs in your total depends on the question: gross units sold excludes it, net revenue includes it. State which one you are computing.

The rule for the whole course: **record the decision, and record how many rows it touched.** A cleaning step with no row count is not reviewable.

In [ ]:
before = len(clean)
dropped_missing = clean['qty'].isna().sum()
dropped_negative = (clean['qty'] < 0).sum()

clean = clean[clean['qty'].notna() & (clean['qty'] > 0)].copy()
clean['qty'] = clean['qty'].astype(int)

print(f'dropped {dropped_missing} row(s) with no quantity')
print(f'dropped {dropped_negative} refund row(s) -- computing GROSS units')
print(f'rows: {before} -> {len(clean)}')

### Fix 4 — text that means the same thing

`Food` and `food`. `RainGear` and `rain-gear`. `Apparel` that is really merch. To a computer these are four or five distinct categories, so every group-by is fragmented and every chart has duplicate bars.

Lowercase and strip first, then map the remaining variants explicitly.

In [ ]:
print('before:', sorted(clean['category'].unique()))

clean['category'] = (clean['category'].str.strip().str.lower()
                     .str.replace('-', '', regex=False))
print('after normalizing case and punctuation:', sorted(clean['category'].unique()))

Case and punctuation get you most of the way. The rest is a business decision that no amount of string handling can make for you: is `apparel` the same category as `merch`? Somebody has to answer that, and the mapping is where you write the answer down.

In [ ]:
CATEGORY_MAP = {
    'apparel': 'merch',      # decision: apparel rolls up into merch
    'raingear': 'raingear',
}
clean['category'] = clean['category'].replace(CATEGORY_MAP)
print('final categories:', sorted(clean['category'].unique()))
print()
print(clean['category'].value_counts())

Do the same for `item`, where the same product is written two ways.

In [ ]:
print('before:', sorted(clean['item'].dropna().unique()))

clean['item'] = clean['item'].str.strip().str.title()
ITEM_MAP = {'Cheese Burger': 'Cheeseburger'}
clean['item'] = clean['item'].replace(ITEM_MAP)
print('after: ', sorted(clean['item'].dropna().unique()))

### Fix 5 — dates in four formats

This column has ISO timestamps, a US-style `09/05/2026 12:40`, a date with no seconds, and one empty cell. `pd.to_datetime` with `format='mixed'` handles the variety, and `errors='coerce'` turns the unparseable ones into `NaT` instead of raising.

In [ ]:
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce', format='mixed')
print(clean['ts'].dtype)
print('unparseable timestamps (NaT):', clean['ts'].isna().sum())
clean[['order_id', 'item', 'ts']]

A real datetime column is what makes the next month of this course possible: you cannot group by hour, join to daily weather, or plot a time series against a text column.

The `NaT` row is a decision again. Keep it if you only need totals; drop it if you are doing anything time-based, because a row with no timestamp cannot be placed on a timeline.

In [ ]:
clean['hour'] = clean['ts'].dt.hour
clean[['order_id', 'item', 'ts', 'hour']]

### Prove it, then measure what changed

State what must now be true. If an assertion fails, you learn it here instead of in a chart three cells later.

In [ ]:
assert clean.duplicated().sum() == 0, 'duplicates remain'
assert clean['price'].dtype == float, 'price is not numeric'
assert clean['qty'].min() >= 1, 'non-positive quantities remain'
assert pd.api.types.is_datetime64_any_dtype(clean['ts']), 'ts is not a datetime'
assert clean['category'].str.islower().all(), 'inconsistent category text'
print('clean:', clean.shape)

Now the part that justifies the whole session. Compute the answer the naive way and the clean way, and look at the gap.

In [ ]:
clean['revenue'] = clean['qty'] * clean['price']

naive_rows = len(df)
naive_units = pd.to_numeric(df['qty'], errors='coerce').fillna(0).sum()
naive_categories = df['category'].nunique()

print(f'rows:        {naive_rows} raw -> {len(clean)} clean')
print(f'units:       {naive_units:.0f} raw -> {clean["qty"].sum()} clean')
print(f'categories:  {naive_categories} raw -> {clean["category"].nunique()} clean')
print(f'revenue:     ${clean["revenue"].sum():.2f} (not computable before cleaning)')

Six apparent categories collapse to three. That is the difference between a chart with six confusing bars and one a manager can read.

### The order matters

This sequence is not arbitrary — each step depends on the last:

1. **Duplicates first.** Cheaper to fix five rows than to fix six and then delete one.
2. **Types next.** You cannot filter on `qty > 0` while `qty` is text.
3. **Then missing and out-of-range values**, now that comparisons work.
4. **Then text normalization**, so grouping is honest.
5. **Then dates**, so anything time-based becomes possible.
6. **Then assert**, so the next person can trust it.

Wednesday you run this whole sequence yourself, and Friday you run it on three hundred rows where you cannot eyeball the answer.

### Practice 1 — revenue by category

Using `clean`, print revenue per category, highest first. Which category would you tell the vendor to stock more of?

In [ ]:
# TODO

### Practice 2 — the refund question

We dropped the refund row and called the result gross units. Recompute **net** units and net revenue with the refund included, starting again from `df`. How much does the answer move, and which number would you put in front of a manager?

In [ ]:
# TODO: start from df, repeat the cleaning, but keep negative quantities

**Which number, and why:** _..._